# Chapitre 14 - Exercices 14.18 à 14.21

Ce notebook traite quatre exercices liés à l'**analyse en composantes indépendantes**
(ICA) et au **clustering spectral** :

- **14.18** : densités gaussiennes « inclinées » pénalisées et conditions de moments ;
- **14.19** : invariance du terme gaussien de la log-vraisemblance par rotation ;
- **14.20** : l'algorithme du point fixe de FastICA vu comme une méthode de Newton ;
- **14.21** : noyau du laplacien d'un graphe et composantes connexes.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from esl.plotting import set_style, COLORS, BLUE, ORANGE, GRAY, SEQ, DIV, TWO_CLASS
from esl import ch14_non_supervise_complements as nsc
set_style()

## Exercice 14.18 : Moments de la densité estimée dans ProDenICA

**Énoncé (reformulé).** Dans l'approche directe de l'ICA (section 14.7.4), on estime la
densité d'une source sous la forme $f = \varphi\,e^{g}$ (gaussienne « inclinée »,
éq. (14.89)), en maximisant la log-vraisemblance régularisée (éq. (14.92)) :
$$\frac1N\sum_{i=1}^N\big[\log\varphi(s_i) + g(s_i)\big] - \int\varphi(t)e^{g(t)}\,dt
- \lambda\int\{g'''(t)\}^2\,dt.$$
La solution $\hat g$ est une spline de lissage quartique, que l'on écrit
$\hat g = \hat q + \hat q_\perp$ où $q(s) = \theta_0 + \theta_1 s + \theta_2 s^2$ est
quadratique (dans le noyau de la pénalité). En examinant les conditions de
stationnarité en $\theta_0, \theta_1, \theta_2$, montrer que $\hat f = \varphi e^{\hat g}$
est une densité, de moyenne nulle et de variance 1. Si l'on utilisait plutôt une pénalité
sur la dérivée seconde, $\int\{g''(t)\}^2dt$, quelle modification simple permettrait de
conserver ces trois conditions de moments ?

### Contexte et notations

- $\varphi(t) = e^{-t^2/2}/\sqrt{2\pi}$ est la densité gaussienne standard.
- Les $s_i = a^Tx_i$ sont les projections des données **blanchies** sur une direction
  unitaire $a$ : les $x_i$ sont centrés et de covariance empirique $I$, donc
  $$\frac1N\sum_i s_i = a^T\bar x = 0,\qquad
  \frac1N\sum_i s_i^2 = a^T\Big(\frac1N\sum_ix_ix_i^T\Big)a = a^Ta = 1.$$
- Le terme $-\int\varphi e^g$ remplace la contrainte « $\int f = 1$ » (on verra qu'il
  l'impose automatiquement à l'optimum).
- Le **noyau de la pénalité** $\int (g''')^2$ est l'ensemble des fonctions de pénalité
  nulle : $g''' = 0$, c'est-à-dire les polynômes de degré $\le 2$.

### Conditions de stationnarité

Notons $F(g)$ le critère. Pour toute fonction $h$ et tout réel $\varepsilon$, si $\hat g$
est un maximum alors $\varepsilon \mapsto F(\hat g + \varepsilon h)$ est maximal en
$\varepsilon = 0$, donc sa dérivée y est nulle. Calculons cette dérivée terme à terme
(en dérivant sous l'intégrale) :

- $\frac{d}{d\varepsilon}\frac1N\sum_i[\log\varphi(s_i) + \hat g(s_i) + \varepsilon h(s_i)]
  = \frac1N\sum_i h(s_i)$ ;
- $\frac{d}{d\varepsilon}\int\varphi e^{\hat g + \varepsilon h}\Big|_{\varepsilon=0}
  = \int\varphi(t)e^{\hat g(t)}h(t)\,dt = \int\hat f(t)h(t)\,dt$ ;
- $\frac{d}{d\varepsilon}\lambda\int(\hat g''' + \varepsilon h''')^2\Big|_{\varepsilon=0}
  = 2\lambda\int\hat g'''h'''$.